# FunctionGemma Fine-Tuning (Colab)

Fine-tune `google/functiongemma-270m-it` to call Cardinal System tools.

**Requirements:** Runtime -> Change runtime type -> T4 GPU

**Steps:**
1. Run Cell 1 (install deps)
2. Run Cell 2 (upload training_data.jsonl)
3. Run Cell 3 (train)
4. Run Cell 4 (download)

In [ ]:
!pip install torch transformers datasets trl accelerate sentencepiece protobuf

In [ ]:
from google.colab import files
print("Upload training_data.jsonl:")
uploaded = files.upload()
DATA_FILE = list(uploaded.keys())[0]
print(f"Loaded: {DATA_FILE}")

In [ ]:
from huggingface_hub import login
login()

import json, torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTTrainer, SFTConfig

MODEL = "google/functiongemma-270m-it"
OUTPUT_DIR = "functiongemma-finetuned"

records = []
with open(DATA_FILE, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))
print(f"Loaded {len(records)} samples")

def format_conversation(sample):
    parts = []
    for msg in sample.get("messages", []):
        role = msg.get("role", "")
        if role == "developer":
            parts.append(msg.get("content", ""))
        elif role == "user":
            parts.append(msg.get("content", ""))
        elif role == "assistant":
            tool_calls = msg.get("tool_calls", [])
            if tool_calls:
                for tc in tool_calls:
                    func = tc.get("function", {})
                    name = func.get("name", "")
                    args = func.get("arguments", {})
                    params = ",".join(
                        f"{k}:<escape>{v}<escape>" for k, v in args.items()
                    )
                    parts.append(
                        f"<start_function_call>call:{name}"
                        f"{{{params}}}<end_function_call>"
                    )
            elif msg.get("content"):
                parts.append(msg["content"])
    return "\n".join(parts)

texts = [format_conversation(r) for r in records]
split_idx = int(len(texts) * 0.9)
train_ds = Dataset.from_dict({"text": texts[:split_idx]})
val_ds = Dataset.from_dict({"text": texts[split_idx:]})
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

tokenizer = AutoTokenizer.from_pretrained(MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL, device_map="auto", torch_dtype="auto"
)
print("Model loaded on:", model.device)

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_length=512,
    num_train_epochs=5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="none",
    fp16=False,
    bf16=True,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
)

print("Starting training...")
trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}/")

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(
    repo_id="skgufranahamed/functiongemma-finetuned",
    repo_type="model",
    exist_ok=True,
)
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id="skgufranahamed/functiongemma-finetuned",
    repo_type="model",
)
print(f"Uploaded to https://huggingface.co/skgufranahamed/functiongemma-finetuned")